<a href="https://colab.research.google.com/github/kroschenko/multi_domain_model/blob/main/Speaker_Aware_Dialogue_Context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 0: SETUP & IMPORTS
!pip install -q transformers accelerate evaluate scikit-learn librosa wandb

import os, gc, json, warnings, shutil, tempfile, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
warnings.filterwarnings('ignore')

# === GOOGLE DRIVE ===
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/erc_article2")
DRIVE_ROOT.mkdir(exist_ok=True)

LOCAL_CACHE = Path("/content/exp2_cache")
LOCAL_CACHE.mkdir(exist_ok=True)

def save_to_drive(filename: str):
    src = Path(f"./{filename}")
    if src.exists():
        shutil.copy2(src, DRIVE_ROOT / filename)
        print(f"Saved to Drive: {filename}")

def backup_cache():
    dst = DRIVE_ROOT / "feat_cache"
    dst.mkdir(exist_ok=True)
    n = 0
    for f in LOCAL_CACHE.rglob("*.npy"):
        shutil.copy2(f, dst / f.name)
        n += 1
    print(f"Backed up {n} feature files to Drive")

# Restore cache if exists
drive_cache = DRIVE_ROOT / "feat_cache"
if drive_cache.exists():
    print("Restoring cache from Drive...")
    for f in drive_cache.rglob("*.npy"):
        dst = LOCAL_CACHE / f.name
        if not dst.exists():
            shutil.copy2(f, dst)
    print(f"Restored {len(list(LOCAL_CACHE.rglob('*.npy')))} files")

# === DEVICE & REPRODUCIBILITY ===
def set_seed(s: int):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float32)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Device: {device} | PyTorch: {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
Mounted at /content/drive
Device: cpu | PyTorch: 2.10.0+cpu


In [2]:
# CELL 1: CONFIGURATION FOR ARTICLE 2
# Speaker-Aware Dialogue Context Encoder
# Research questions:
#   RQ1: Does speaker-aware encoding improve F1 vs. speaker-agnostic?
#   RQ2: How does context window size affect performance?
#   RQ3: Does learnable temporal decay outperform fixed?
#   RQ4: Turn-taking signals as explicit features?

@dataclass
class Exp2Config:
    # Data
    max_utterances: int = 7          # context window (ablation: 1,3,5,7,9)
    max_seq_len: int = 64            # per-utterance token length

    # Model
    text_model: str = "microsoft/deberta-v3-base"
    freeze_text: bool = True
    hidden_dim: int = 256
    dropout: float = 0.2
    num_speakers: int = 30           # MELD has ~260 chars → clip to top-30 + UNK
    n_transformer_layers: int = 2
    n_heads: int = 4

    # Speaker-aware features
    use_speaker_emb: bool = True     # ablation: turn off
    use_role_emb: bool = True        # same/diff speaker binary
    use_pos_emb: bool = True         # position in dialogue
    use_turn_taking: bool = True     # speaker-change binary feature
    learnable_decay: bool = True     # ablation: fixed vs learnable λ
    n_decay_heads: int = 4           # multi-head temporal decay

    # Training
    batch_size: int = 16
    epochs: int = 6
    lr: float = 2e-4
    weight_decay: float = 0.01
    early_stopping_patience: int = 4
    warmup_ratio: float = 0.1

    # Experiment
    n_seeds: int = 3
    experiment_name: str = "exp2_speaker_dialogue_context"
    cache_dir: str = str(LOCAL_CACHE)

    # Baselines to compare
    baselines: List[str] = None
    def __post_init__(self):
        if self.baselines is None:
            self.baselines = [
                "TextOnly_NoContext",
                "Context_NoSpeaker",
                "Context_FixedDecay",
                "Context_NoRoleEmb",
                "Context_NoTurnTaking",
                "Full_SpeakerAware",      # our model
                "DialogueRNN_baseline",
            ]

config = Exp2Config()
print(f"Config: {config.experiment_name}")
print(f"Context window: {config.max_utterances} utterances")
print(f"Ablations: {config.baselines}")

Config: exp2_speaker_dialogue_context
Context window: 7 utterances
Ablations: ['TextOnly_NoContext', 'Context_NoSpeaker', 'Context_FixedDecay', 'Context_NoRoleEmb', 'Context_NoTurnTaking', 'Full_SpeakerAware', 'DialogueRNN_baseline']


In [3]:
# CELL 2: TEXT FEATURE EXTRACTION WITH CACHING
# Используем DeBERTa CLS-токен для каждого utterance отдельно,
# кэшируем на диск чтобы не перекодировать в каждом эксперименте.

class UtteranceEncoder:
    """Pre-encodes utterances with DeBERTa, caches to disk."""
    def __init__(self, model_name: str = "microsoft/deberta-v3-base",
                 cache_dir: Path = LOCAL_CACHE, max_len: int = 64):
        print(f"Loading {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).eval().to(device)
        for p in self.model.parameters():
            p.requires_grad = False
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.max_len = max_len
        self.dim = 768

    @torch.no_grad()
    def encode(self, text: str, uid: str) -> np.ndarray:
        cache_file = self.cache_dir / f"text_{uid}.npy"
        if cache_file.exists():
            return np.load(cache_file)

        tok = self.tokenizer(
            text, padding='max_length', truncation=True,
            max_length=self.max_len, return_tensors='pt'
        )
        tok = {k: v.to(device) for k, v in tok.items()}
        out = self.model(**tok).last_hidden_state[:, 0].squeeze().cpu().numpy()
        np.save(cache_file, out.astype(np.float32))
        return out.astype(np.float32)

    def encode_dataframe(self, df: pd.DataFrame) -> None:
        """Pre-encode all utterances in a DataFrame."""
        print(f"Pre-encoding {len(df)} utterances...")
        for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
            uid = f"dia{int(row['Dialogue_ID'])}_utt{int(row['Utterance_ID'])}"
            text = str(row.get('Utterance', row.get('text', '')))
            self.encode(text, uid)
        print("Done encoding.")


utterance_encoder = UtteranceEncoder(
    model_name=config.text_model,
    cache_dir=LOCAL_CACHE,
    max_len=config.max_seq_len
)
print("Utterance encoder ready.")

Loading microsoft/deberta-v3-base...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Utterance encoder ready.


In [ ]:
# CELL 3: DATA LOADING + SPEAKER-AWARE DATASET

try:
    import kagglehub
    print("Downloading MELD...")
    path = kagglehub.dataset_download("bhandariprakanda/meld-emotion-recognition")
    video_base = os.path.join(path, "MELD.Raw", "MELD.Raw")
    csv_path = os.path.join(path, "JSON files", "JSON files", "Updated CSV")
    train_df = pd.read_csv(os.path.join(csv_path, "train_sent_emo_cleaned.csv"))
    test_df  = pd.read_csv(os.path.join(csv_path, "test_sent_emo_cleaned.csv"))
    print(f"MELD loaded: train={len(train_df)}, test={len(test_df)}")
except Exception as e:
    print(f"Using dummy data ({e})")
    rng = np.random.default_rng(42)
    emotions = ["neutral","joy","sadness","anger","surprise","disgust","fear"]
    speakers = ["Monica","Rachel","Phoebe","Ross","Chandler","Joey"]
    n = 400
    train_df = pd.DataFrame({
        "Dialogue_ID":   rng.integers(0, 40, n),
        "Utterance_ID":  rng.integers(0, 15, n),
        "Utterance":     [f"utterance {i}" for i in range(n)],
        "Speaker":       rng.choice(speakers, n),
        "Emotion":       rng.choice(emotions, n),
    })
    test_df = train_df.iloc[:80].copy()

EMOTIONS = ["neutral","joy","sadness","anger","surprise","disgust","fear"]
train_df = train_df[train_df["Emotion"].isin(EMOTIONS)].reset_index(drop=True)
test_df  = test_df[test_df["Emotion"].isin(EMOTIONS)].reset_index(drop=True)

# Val split
val_df = train_df.groupby("Emotion", group_keys=False).apply(
    lambda x: x.sample(frac=0.15, random_state=42)
).reset_index(drop=True)
train_df = train_df.drop(val_df.index).reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Emotions: {EMOTIONS}")
print(f"Speakers (top-10): {train_df['Speaker'].value_counts().head(10).to_dict()}")

# Pre-encode ALL utterances once
utterance_encoder.encode_dataframe(train_df)
utterance_encoder.encode_dataframe(val_df)
utterance_encoder.encode_dataframe(test_df)
print("All utterances encoded and cached.")

Using Colab cache for faster access to the 'meld-emotion-recognition' dataset.
MELD loaded: train=9988, test=2610
Train: 8491 | Val: 1497 | Test: 2610
Emotions: ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'disgust', 'fear']
Speakers (top-10): {'Joey': 1251, 'Rachel': 1239, 'Ross': 1220, 'Phoebe': 1166, 'Monica': 1130, 'Chandler': 1065, 'Janice': 51, 'Carol': 45, 'Tag': 37, 'Emily': 37}
Pre-encoding 8491 utterances...


  0%|          | 15/8491 [01:36<10:55:32,  4.64s/it]

In [ ]:
# CELL 4: SPEAKER-AWARE DIALOGUE DATASET

class DialogueContextDataset(Dataset):
    """
    Returns each utterance WITH its dialogue context window.
    Each sample = dict with:
      - context_embs: [max_utt, 768] — DeBERTa CLS for each context utterance
      - context_speakers: [max_utt] — speaker IDs (0..num_speakers)
      - role_mask: [max_utt] — 1 if same speaker as target, else 0
      - turn_taking: [max_utt] — 1 if speaker changed vs previous utterance
      - pos_ids: [max_utt] — position indices
      - current_idx: int — index of target utterance in window
      - label: int
      - uid: str
    """
    def __init__(self, df: pd.DataFrame, config: Exp2Config,
                 cache_dir: Path = LOCAL_CACHE):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.cache_dir = Path(cache_dir)

        self.emo2idx = {e: i for i, e in enumerate(EMOTIONS)}

        # Build speaker map: top-(num_speakers-1) + UNK=0
        top_spk = df['Speaker'].value_counts().head(config.num_speakers - 1).index.tolist()
        self.spk2idx = {s: i+1 for i, s in enumerate(top_spk)}  # 0 = UNK

        # Pre-compute context windows
        self._build_windows()

    def _build_windows(self):
        self.windows = []
        W = self.config.max_utterances
        half = W // 2

        for idx, row in self.df.iterrows():
            dia = row['Dialogue_ID']
            utt = row['Utterance_ID']

            # Get context rows (sorted)
            mask = (self.df['Dialogue_ID'] == dia) & \
                   (self.df['Utterance_ID'] >= utt - half) & \
                   (self.df['Utterance_ID'] <= utt + half)
            ctx = self.df[mask].sort_values('Utterance_ID')

            # Find target position in context
            target_pos = (ctx['Utterance_ID'].values == utt).argmax()

            rows = ctx.to_dict('records')
            self.windows.append((rows, target_pos, row['Emotion']))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        rows, target_pos, emotion = self.windows[idx]
        W = self.config.max_utterances
        cfg = self.config

        # Load cached embeddings
        embs, speaker_ids, turn_taking = [], [], []
        prev_spk = None
        for r in rows:
            uid = f"dia{int(r['Dialogue_ID'])}_utt{int(r['Utterance_ID'])}"
            cache_f = self.cache_dir / f"text_{uid}.npy"
            if cache_f.exists():
                emb = np.load(cache_f).astype(np.float32)
            else:
                emb = np.zeros(768, dtype=np.float32)
            embs.append(emb)

            spk = r.get('Speaker', 'UNK')
            sid = self.spk2idx.get(spk, 0)
            speaker_ids.append(sid)

            # Turn-taking: did speaker change?
            changed = 1 if (prev_spk is not None and spk != prev_spk) else 0
            turn_taking.append(changed)
            prev_spk = spk

        # Role mask: same speaker as target
        target_spk = speaker_ids[target_pos] if target_pos < len(speaker_ids) else 0
        role_mask = [1 if s == target_spk else 0 for s in speaker_ids]

        # Pad/truncate to W
        def pad_list(lst, pad_val, dtype):
            lst = lst[:W]
            while len(lst) < W:
                lst.append(pad_val)
            return lst

        embs_arr = np.stack([embs[i] if i < len(embs) else np.zeros(768, dtype=np.float32)
                             for i in range(min(len(embs), W))] +
                            [np.zeros(768, dtype=np.float32)] * max(0, W - len(embs)))

        sp_arr   = pad_list(speaker_ids, 0, torch.long)
        role_arr = pad_list(role_mask,   0, torch.long)
        tt_arr   = pad_list(turn_taking, 0, torch.long)
        pos_arr  = list(range(W))
        cur_idx  = min(target_pos, W - 1)

        return {
            "context_embs":    torch.from_numpy(embs_arr[:W]).float(),  # [W, 768]
            "context_speakers": torch.tensor(sp_arr[:W],   dtype=torch.long),
            "role_mask":       torch.tensor(role_arr[:W],  dtype=torch.long),
            "turn_taking":     torch.tensor(tt_arr[:W],    dtype=torch.long),
            "pos_ids":         torch.tensor(pos_arr[:W],   dtype=torch.long),
            "current_idx":     torch.tensor(cur_idx,       dtype=torch.long),
            "label":           torch.tensor(self.emo2idx[emotion], dtype=torch.long),
            "uid": f"dia{int(rows[target_pos if target_pos < len(rows) else 0]['Dialogue_ID'])}"
                   f"_utt{int(rows[target_pos if target_pos < len(rows) else 0]['Utterance_ID'])}",
        }


def make_loaders(train_df, val_df, test_df, config):
    train_ds = DialogueContextDataset(train_df, config)
    val_ds   = DialogueContextDataset(val_df,   config)
    test_ds  = DialogueContextDataset(test_df,  config)

    kw = dict(collate_fn=None, num_workers=2, pin_memory=True)
    dl_tr = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True,  **kw)
    dl_vl = DataLoader(val_ds,   batch_size=config.batch_size, shuffle=False, **kw)
    dl_ts = DataLoader(test_ds,  batch_size=config.batch_size, shuffle=False, **kw)
    return dl_tr, dl_vl, dl_ts, train_ds, val_ds, test_ds

dl_tr, dl_vl, dl_ts, train_ds, val_ds, test_ds = make_loaders(train_df, val_df, test_df, config)
print(f"Loaders ready: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

# Sanity check
b = next(iter(dl_tr))
print(f"Batch keys: {list(b.keys())}")
print(f"context_embs: {b['context_embs'].shape}")
print(f"context_speakers: {b['context_speakers'].shape}")
print(f"label distribution: {b['label'].bincount(minlength=7).tolist()}")

In [ ]:
# CELL 5: SPEAKER-AWARE DIALOGUE CONTEXT ENCODER (core contribution)

class LearnableTemporalDecay(nn.Module):
    """
    Multi-head learnable temporal decay pooling.
    w_ih = exp(−λ_h · |i − t|), λ_h learned per head.
    Output: weighted avg across heads → projected to hidden_dim.

    Fixed decay baseline: replace with fixed λ=1.0.
    """
    def __init__(self, hidden_dim: int, n_heads: int = 4):
        super().__init__()
        self.n_heads = n_heads
        # One λ per head, initialized to small positive values
        self.log_lambda = nn.Parameter(torch.zeros(n_heads))  # λ = exp(log_λ) > 0
        self.head_proj = nn.Linear(hidden_dim, hidden_dim * n_heads)
        self.out_proj   = nn.Linear(hidden_dim * n_heads, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor, current_idx: torch.Tensor,
                fixed_lambda: Optional[float] = None) -> torch.Tensor:
        """
        x: [B, W, D]
        current_idx: [B]
        Returns: [B, D]
        """
        B, W, D = x.shape

        # Multi-head projection
        xh = self.head_proj(x)  # [B, W, D*H]
        xh = xh.view(B, W, self.n_heads, D)  # [B, W, H, D]

        # Compute decay weights per head
        positions = torch.arange(W, device=x.device).unsqueeze(0)  # [1, W]
        dist = (positions - current_idx.unsqueeze(1)).abs().float()  # [B, W]

        if fixed_lambda is not None:
            lam = torch.full((self.n_heads,), fixed_lambda, device=x.device)
        else:
            lam = torch.exp(self.log_lambda)  # [H], always positive

        # weights: [B, W, H]
        weights = torch.exp(-lam.unsqueeze(0).unsqueeze(0) * dist.unsqueeze(-1))
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)  # normalize

        # Weighted pool per head: [B, H, D]
        pooled = (xh * weights.unsqueeze(-1)).sum(dim=1)  # [B, H, D]
        pooled = pooled.view(B, self.n_heads * D)

        out = self.out_proj(pooled)  # [B, D]
        return self.norm(out + x[:, 0])  # residual from first position


class SpeakerAwareDialogueEncoder(nn.Module):
    """
    Article #2 core model.
    Ablation flags:
      use_speaker_emb, use_role_emb, use_pos_emb,
      use_turn_taking, learnable_decay
    """
    def __init__(self, config: Exp2Config, num_classes: int = 7):
        super().__init__()
        self.config = config
        D = config.hidden_dim

        # Project DeBERTa [768] → hidden_dim
        self.text_proj = nn.Linear(768, D)

        # Optional embeddings
        if config.use_speaker_emb:
            self.speaker_emb = nn.Embedding(config.num_speakers + 1, D, padding_idx=0)
        if config.use_role_emb:
            self.role_emb = nn.Embedding(2, D)
        if config.use_pos_emb:
            self.pos_emb = nn.Embedding(config.max_utterances, D)
        if config.use_turn_taking:
            self.turn_emb = nn.Embedding(2, D // 4)
            self.turn_proj = nn.Linear(D + D // 4, D)

        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=D, nhead=config.n_heads,
            dim_feedforward=D * 2, dropout=config.dropout,
            batch_first=True, norm_first=True  # Pre-LN for stability
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=config.n_transformer_layers)

        # Temporal decay pooling
        self.decay_pool = LearnableTemporalDecay(D, n_heads=config.n_decay_heads)

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(D, D // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(D // 2, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, batch: Dict, training: bool = False) -> Tuple[torch.Tensor, Dict]:
        cfg = self.config
        ctx_embs    = batch['context_embs'].to(device)       # [B, W, 768]
        ctx_speakers = batch['context_speakers'].to(device)  # [B, W]
        role_mask   = batch['role_mask'].to(device)          # [B, W]
        turn_taking = batch['turn_taking'].to(device)        # [B, W]
        pos_ids     = batch['pos_ids'].to(device)            # [B, W]
        current_idx = batch['current_idx'].to(device)        # [B]

        B, W, _ = ctx_embs.shape

        # Project text embeddings
        x = self.text_proj(ctx_embs)  # [B, W, D]

        # Add optional embeddings
        if cfg.use_speaker_emb:
            spk = torch.clamp(ctx_speakers, 0, cfg.num_speakers)
            x = x + self.speaker_emb(spk)

        if cfg.use_role_emb:
            x = x + self.role_emb(role_mask)

        if cfg.use_pos_emb:
            x = x + self.pos_emb(pos_ids)

        if cfg.use_turn_taking:
            tt_feat = self.turn_emb(turn_taking)              # [B, W, D//4]
            x = self.turn_proj(torch.cat([x, tt_feat], dim=-1))

        # Causal mask (don't attend to future utterances beyond target)
        tgt_idx = current_idx[0].item()
        causal_mask = torch.zeros(W, W, device=device, dtype=torch.bool)
        if tgt_idx + 1 < W:
            causal_mask[:, tgt_idx+1:] = True  # mask future

        # Transformer
        x = self.transformer(x, src_key_padding_mask=None)  # [B, W, D]

        # Temporal decay pooling
        fixed_lam = None if cfg.learnable_decay else 1.0
        pooled = self.decay_pool(x, current_idx, fixed_lambda=fixed_lam)  # [B, D]

        # Classify
        logits = self.classifier(pooled)  # [B, num_classes]

        # Return attention-like info for analysis
        info = {
            'decay_lambdas': torch.exp(self.decay_pool.log_lambda).detach(),
        }
        return logits, info


# Quick sanity check
model_test = SpeakerAwareDialogueEncoder(config, num_classes=7).to(device).float()
b = next(iter(dl_tr))
b = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in b.items()}
with torch.no_grad():
    logits, info = model_test(b)
print(f"Forward OK: logits={logits.shape}, lambdas={info['decay_lambdas'].tolist()}")
del model_test
torch.cuda.empty_cache()

In [ ]:
# CELL 6: TRAINING LOOP WITH WARMUP + EARLY STOPPING

from transformers import get_cosine_schedule_with_warmup

def train_exp2(model: nn.Module, dl_tr: DataLoader, dl_vl: DataLoader,
               config: Exp2Config, seed: int,
               class_weights: torch.Tensor) -> Tuple[nn.Module, float, List[Dict]]:

    model = model.float().to(device)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config.lr, weight_decay=config.weight_decay
    )
    total_steps = config.epochs * len(dl_tr)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * config.warmup_ratio),
        num_training_steps=total_steps
    )
    scaler = GradScaler()
    cw = class_weights.to(device)

    best_f1, best_state, patience = 0.0, None, 0
    history = []

    for epoch in range(config.epochs):
        model.train()
        epoch_loss = 0.0

        for batch in tqdm(dl_tr, desc=f"Seed {seed} Ep {epoch+1}", leave=False):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            optimizer.zero_grad()

            with autocast():
                logits, _ = model(batch, training=True)
                loss = F.cross_entropy(logits.float(), batch['label'], weight=cw)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            epoch_loss += loss.item()

        # === Validation ===
        model.eval()
        preds, trues, all_lambdas = [], [], []
        with torch.no_grad():
            for batch in dl_vl:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in batch.items()}
                logits, info = model(batch, training=False)
                preds.extend(logits.argmax(1).cpu().numpy())
                trues.extend(batch['label'].cpu().numpy())
                all_lambdas.append(info['decay_lambdas'].cpu().numpy())

        f1 = f1_score(trues, preds, average='macro', zero_division=0)
        mean_lambdas = np.mean(all_lambdas, axis=0)

        history.append({
            'epoch': epoch + 1,
            'loss': epoch_loss / len(dl_tr),
            'val_f1': f1,
            'decay_lambdas': mean_lambdas.tolist()
        })

        print(f"  Epoch {epoch+1}: loss={epoch_loss/len(dl_tr):.4f} | val_F1={f1:.4f} "
              f"| λ={mean_lambdas.round(3).tolist()}")

        if f1 > best_f1:
            best_f1 = f1
            patience = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience += 1
            if patience >= config.early_stopping_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    if best_state:
        model.load_state_dict(best_state)
    torch.cuda.empty_cache()
    gc.collect()
    return model, best_f1, history


def evaluate(model: nn.Module, dl: DataLoader) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            logits, _ = model(batch, training=False)
            preds.extend(logits.argmax(1).cpu().numpy())
            trues.extend(batch['label'].cpu().numpy())
    return np.array(preds), np.array(trues)


# Compute class weights
train_labels = np.array([s['label'].item() for s in train_ds])
cw_np = compute_class_weight('balanced', classes=np.arange(len(EMOTIONS)), y=train_labels)
CLASS_WEIGHTS = torch.tensor(cw_np, dtype=torch.float32)
print(f"Class weights: {dict(zip(EMOTIONS, cw_np.round(2)))}")

In [ ]:
# CELL 7: FULL ABLATION SUITE (RQ1–RQ4)

ABLATION_CONFIGS = {
    "TextOnly_NoContext": dict(
        max_utterances=1, use_speaker_emb=False, use_role_emb=False,
        use_pos_emb=False, use_turn_taking=False, learnable_decay=False
    ),
    "Context_NoSpeaker": dict(
        use_speaker_emb=False, use_role_emb=False, use_turn_taking=False
    ),
    "Context_FixedDecay": dict(
        learnable_decay=False
    ),
    "Context_NoRoleEmb": dict(
        use_role_emb=False
    ),
    "Context_NoTurnTaking": dict(
        use_turn_taking=False
    ),
    "Full_SpeakerAware": dict(),  # full model, no overrides
}

# Context window ablation (RQ2)
WINDOW_SIZES = [1, 3, 5, 7, 9]

all_ablation_results = {}

for variant_name, overrides in ABLATION_CONFIGS.items():
    print(f"\n{'='*50}")
    print(f"Running: {variant_name}")
    print(f"Overrides: {overrides}")

    seed_f1s = []
    seed_preds = []
    seed_trues = None

    for seed in range(42, 42 + config.n_seeds):
        set_seed(seed)

        # Build variant config
        variant_cfg_dict = asdict(config)
        variant_cfg_dict.update(overrides)
        variant_cfg = Exp2Config(**{k: v for k, v in variant_cfg_dict.items()
                                    if k in Exp2Config.__dataclass_fields__})

        # Rebuild loaders if window size changed
        if 'max_utterances' in overrides:
            _dl_tr, _dl_vl, _dl_ts, _tr_ds, _, _ = make_loaders(
                train_df, val_df, test_df, variant_cfg)
        else:
            _dl_tr, _dl_vl, _dl_ts, _tr_ds = dl_tr, dl_vl, dl_ts, train_ds

        model_v = SpeakerAwareDialogueEncoder(variant_cfg, num_classes=len(EMOTIONS)).to(device).float()

        _, best_f1, history = train_exp2(
            model_v, _dl_tr, _dl_vl, variant_cfg, seed, CLASS_WEIGHTS
        )

        preds, trues = evaluate(model_v, _dl_ts)
        f1_test = f1_score(trues, preds, average='macro', zero_division=0)
        wf1_test = f1_score(trues, preds, average='weighted', zero_division=0)

        seed_f1s.append(f1_test)
        seed_preds.append(preds)
        if seed_trues is None:
            seed_trues = trues

        print(f"  Seed {seed}: Test Macro-F1={f1_test:.3f} | Weighted={wf1_test:.3f}")

        del model_v
        torch.cuda.empty_cache()
        gc.collect()

    # Wilcoxon vs Full model (if available)
    p_val = "—"
    if variant_name != "Full_SpeakerAware" and "Full_SpeakerAware" in all_ablation_results:
        full_preds_list = all_ablation_results["Full_SpeakerAware"]["raw_preds"]
        diffs = []
        for vp, fp in zip(seed_preds, full_preds_list):
            if len(vp) == len(fp) and seed_trues is not None:
                d = (vp == seed_trues).astype(float) - (fp == seed_trues).astype(float)
                diffs.append(d)
        if diffs:
            all_diffs = np.concatenate(diffs)
            try:
                _, pv = wilcoxon(all_diffs, alternative='less', method='approx')
                p_val = f"{pv:.4f}"
            except Exception:
                p_val = "N/A"

    all_ablation_results[variant_name] = {
        "macro_f1": f"{np.mean(seed_f1s):.3f} ± {np.std(seed_f1s):.3f}",
        "mean_f1": np.mean(seed_f1s),
        "std_f1": np.std(seed_f1s),
        "p_value_vs_full": p_val,
        "raw_preds": seed_preds,
        "true_labels": seed_trues,
    }
    print(f"  → {variant_name}: {all_ablation_results[variant_name]['macro_f1']} (p={p_val})")

# Save ablation table
ablation_df = pd.DataFrame([
    {"Model": k,
     "Macro-F1": v["macro_f1"],
     "p-value vs Full": v["p_value_vs_full"]}
    for k, v in all_ablation_results.items()
])
ablation_df.to_csv("exp2_ablation_results.csv", index=False)
print("\nAblation table:")
print(ablation_df.to_string(index=False))
save_to_drive("exp2_ablation_results.csv")

In [ ]:
# CELL 8: CONTEXT WINDOW SIZE ABLATION (RQ2)
# Runs 1 seed per window size to save time; for paper use n_seeds=3

window_results = {}

for W in WINDOW_SIZES:
    print(f"\nWindow size W={W}")
    w_cfg_dict = asdict(config)
    w_cfg_dict['max_utterances'] = W
    w_cfg = Exp2Config(**{k: v for k, v in w_cfg_dict.items()
                          if k in Exp2Config.__dataclass_fields__})

    set_seed(42)
    _dl_tr, _dl_vl, _dl_ts, _, _, _ = make_loaders(train_df, val_df, test_df, w_cfg)

    m = SpeakerAwareDialogueEncoder(w_cfg, num_classes=len(EMOTIONS)).to(device).float()
    _, best_f1, hist = train_exp2(m, _dl_tr, _dl_vl, w_cfg, 42, CLASS_WEIGHTS)
    preds, trues = evaluate(m, _dl_ts)
    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    window_results[W] = {"f1": f1, "best_val_f1": best_f1}
    print(f"  W={W} → Test F1={f1:.3f}")

    del m; torch.cuda.empty_cache(); gc.collect()

# Save
window_df = pd.DataFrame([{"window": k, "macro_f1": v["f1"]} for k, v in window_results.items()])
window_df.to_csv("exp2_window_ablation.csv", index=False)
save_to_drive("exp2_window_ablation.csv")
print(window_df.to_string(index=False))

In [ ]:
# CELL 9: VISUALIZATIONS FOR PAPER

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import confusion_matrix

VIZ_DIR = Path("./exp2_viz")
VIZ_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'figure.dpi': 300, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'figure.facecolor': 'white'
})

# ── Fig 1: Ablation bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
variants = list(all_ablation_results.keys())
means = [all_ablation_results[v]["mean_f1"] for v in variants]
stds  = [all_ablation_results[v]["std_f1"]  for v in variants]
colors = ['#4C72B0' if v == "Full_SpeakerAware" else '#8DA0CB' for v in variants]

bars = ax.barh(variants, means, xerr=stds, color=colors,
               edgecolor='white', linewidth=0.5, capsize=3, height=0.55)

# Annotate bars
for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_width() + s + 0.003, bar.get_y() + bar.get_height()/2,
            f'{m:.3f}', va='center', fontsize=9)

ax.set_xlabel('Macro-F1 (Test set)')
ax.set_title('Ablation study: Speaker-Aware Dialogue Context Encoder')
ax.axvline(means[-1], color='#4C72B0', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlim(max(0, min(means) - 0.05), max(means) + 0.06)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig1_ablation_bar.pdf")
plt.savefig(VIZ_DIR / "fig1_ablation_bar.png")
plt.show()
print("Saved: fig1_ablation_bar")

# ── Fig 2: Context window curve ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 3.5))
ws = list(window_results.keys())
f1s = [window_results[w]["f1"] for w in ws]
ax.plot(ws, f1s, 'o-', color='#4C72B0', linewidth=2, markersize=7)
for w, f in zip(ws, f1s):
    ax.annotate(f'{f:.3f}', (w, f), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=9)
ax.set_xlabel('Context window size (utterances)')
ax.set_ylabel('Macro-F1')
ax.set_title('Effect of context window size on emotion recognition')
ax.set_xticks(ws)
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig2_window_curve.pdf")
plt.savefig(VIZ_DIR / "fig2_window_curve.png")
plt.show()
print("Saved: fig2_window_curve")

# ── Fig 3: Confusion matrix (Full model, last seed) ────────────────────────────
full_preds = all_ablation_results["Full_SpeakerAware"]["raw_preds"][-1]
full_trues = all_ablation_results["Full_SpeakerAware"]["true_labels"]

cm = confusion_matrix(full_trues, full_preds, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS,
            linewidths=0.3, linecolor='white', ax=ax, vmin=0, vmax=1)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized confusion matrix — Full Speaker-Aware model')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig3_confusion_matrix.pdf")
plt.savefig(VIZ_DIR / "fig3_confusion_matrix.png")
plt.show()
print("Saved: fig3_confusion_matrix")

# ── Fig 4: F1 per emotion (Full vs NoSpeaker) ─────────────────────────────────
from sklearn.metrics import f1_score

def per_class_f1(preds, trues, classes):
    return [f1_score(trues == i, preds == i, zero_division=0) for i in range(len(classes))]

full_p = all_ablation_results["Full_SpeakerAware"]["raw_preds"][-1]
full_t = all_ablation_results["Full_SpeakerAware"]["true_labels"]
nosp_p = all_ablation_results["Context_NoSpeaker"]["raw_preds"][-1]
nosp_t = all_ablation_results["Context_NoSpeaker"]["true_labels"]

f1_full = per_class_f1(full_p, full_t, EMOTIONS)
f1_nosp = per_class_f1(nosp_p, nosp_t, EMOTIONS)

x = np.arange(len(EMOTIONS))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - w/2, f1_full, w, label='Full Speaker-Aware', color='#4C72B0')
ax.bar(x + w/2, f1_nosp, w, label='No Speaker Info',   color='#C44E52', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(EMOTIONS, rotation=20, ha='right')
ax.set_ylabel('F1 score')
ax.set_title('Per-emotion F1: Full model vs. speaker-agnostic baseline')
ax.legend()
ax.set_ylim(0, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig4_per_emotion_f1.pdf")
plt.savefig(VIZ_DIR / "fig4_per_emotion_f1.png")
plt.show()
print("Saved: fig4_per_emotion_f1")

# ── Fig 5: Learned decay lambdas per head ─────────────────────────────────────
# Train a quick model to get lambdas
set_seed(42)
lambda_model = SpeakerAwareDialogueEncoder(config, num_classes=len(EMOTIONS)).to(device).float()
_, _, hist = train_exp2(lambda_model, dl_tr, dl_vl, config, 42, CLASS_WEIGHTS)

lambda_vals = torch.exp(lambda_model.decay_pool.log_lambda).detach().cpu().numpy()
epoch_lambdas = [h['decay_lambdas'] for h in hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Bar: final lambda per head
heads = [f"Head {i+1}" for i in range(config.n_decay_heads)]
ax1.bar(heads, lambda_vals, color='#4C72B0', edgecolor='white')
ax1.set_ylabel('λ (decay rate)')
ax1.set_title('Learned temporal decay rate per head')
ax1.axhline(1.0, color='gray', linestyle='--', label='Fixed baseline (λ=1)')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Line: lambdas across training
for h in range(config.n_decay_heads):
    vals = [ep[h] for ep in epoch_lambdas]
    ax2.plot(range(1, len(vals)+1), vals, 'o-', label=f"Head {h+1}", linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('λ')
ax2.set_title('Temporal decay rates during training')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig5_decay_lambdas.pdf")
plt.savefig(VIZ_DIR / "fig5_decay_lambdas.png")
plt.show()
print("Saved: fig5_decay_lambdas")

del lambda_model; torch.cuda.empty_cache(); gc.collect()
print(f"\nAll figures saved to {VIZ_DIR}")

In [ ]:
# CELL 10: F1 PER TURN POSITION IN DIALOGUE
# Paper Section 4.3: "Does speaker context help more at turn N?"

def analyze_turn_position(model_preds, model_trues, test_df_reset, max_turn=15):
    """
    Compute per-turn F1: group test samples by utterance position in dialogue.
    """
    test_df_reset = test_df_reset.reset_index(drop=True)
    assert len(model_preds) == len(test_df_reset), \
        f"Length mismatch: {len(model_preds)} preds vs {len(test_df_reset)} rows"

    results = []
    for turn in range(max_turn + 1):
        mask = test_df_reset['Utterance_ID'] == turn
        if mask.sum() < 5:
            continue
        idxs = np.where(mask.values)[0]
        p = model_preds[idxs]
        t = model_trues[idxs]
        f1 = f1_score(t, p, average='macro', zero_division=0)
        results.append({'turn': turn, 'f1': f1, 'n': int(mask.sum())})

    return pd.DataFrame(results)

# Use full model and no-speaker baseline predictions
turn_full = analyze_turn_position(
    all_ablation_results["Full_SpeakerAware"]["raw_preds"][-1],
    all_ablation_results["Full_SpeakerAware"]["true_labels"],
    test_df
)
turn_nosp = analyze_turn_position(
    all_ablation_results["Context_NoSpeaker"]["raw_preds"][-1],
    all_ablation_results["Context_NoSpeaker"]["true_labels"],
    test_df
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(turn_full['turn'], turn_full['f1'], 'o-', color='#4C72B0',
        linewidth=2, label='Full Speaker-Aware')
ax.plot(turn_nosp['turn'], turn_nosp['f1'], 's--', color='#C44E52',
        linewidth=2, alpha=0.8, label='No Speaker Info')
ax.fill_between(turn_full['turn'], turn_full['f1'], turn_nosp['f1'],
                alpha=0.15, color='#4C72B0')
ax.set_xlabel('Utterance position in dialogue (turn index)')
ax.set_ylabel('Macro-F1')
ax.set_title('F1 per dialogue turn: speaker-aware vs. baseline')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.35)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIZ_DIR / "fig6_f1_per_turn.pdf")
plt.savefig(VIZ_DIR / "fig6_f1_per_turn.png")
plt.show()

turn_full.to_csv("exp2_turn_analysis.csv", index=False)
save_to_drive("exp2_turn_analysis.csv")
print("Saved: fig6_f1_per_turn + turn analysis CSV")

In [ ]:
# CELL 11: LATEX TABLE + FINAL SUMMARY

def generate_latex_table_exp2(results: dict) -> str:
    lines = [
        r"\begin{table}[t]",
        r"\centering\small",
        r"\caption{Ablation study: Speaker-Aware Dialogue Context Encoder on MELD test set.}",
        r"\label{tab:exp2_ablation}",
        r"\begin{tabular}{lccc}",
        r"\toprule",
        r"\textbf{Model} & \textbf{Macro-F1} & \textbf{$\Delta$ vs Full} & \textbf{$p$-value} \\",
        r"\midrule",
    ]
    full_f1 = results["Full_SpeakerAware"]["mean_f1"]
    for name, v in results.items():
        f1 = v["mean_f1"]
        std = v["std_f1"]
        delta = f"+{f1-full_f1:.3f}" if f1 > full_f1 else f"{f1-full_f1:.3f}"
        if name == "Full_SpeakerAware":
            delta = "—"
        pval = v.get("p_value_vs_full", "—")
        name_tex = name.replace("_", "\\_")
        if name == "Full_SpeakerAware":
            lines.append(r"\textbf{" + f"{name_tex}" + r"} & " +
                         r"\textbf{" + f"{f1:.3f} $\\pm$ {std:.3f}" + r"} & " +
                         f"{delta} & {pval} \\\\")
        else:
            lines.append(f"{name_tex} & {f1:.3f} $\\pm$ {std:.3f} & {delta} & {pval} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)

latex_str = generate_latex_table_exp2(all_ablation_results)
print("LaTeX Table:\n")
print(latex_str)

# Save
with open("exp2_latex_table.tex", "w") as f:
    f.write(latex_str)
save_to_drive("exp2_latex_table.tex")

# === FINAL SUMMARY ===
print("\n" + "="*60)
print("EXPERIMENT 2 SUMMARY: Speaker-Aware Dialogue Context")
print("="*60)
print(f"\nRQ1 — Does speaker info help?")
full_f1  = all_ablation_results['Full_SpeakerAware']['mean_f1']
nosp_f1  = all_ablation_results['Context_NoSpeaker']['mean_f1']
print(f"  Full model:       {full_f1:.3f}")
print(f"  No speaker info:  {nosp_f1:.3f}")
print(f"  Delta:            {full_f1 - nosp_f1:+.3f}")

print(f"\nRQ2 — Optimal context window?")
best_w = max(window_results, key=lambda w: window_results[w]['f1'])
print(f"  Best window size: W={best_w} (F1={window_results[best_w]['f1']:.3f})")

print(f"\nRQ3 — Learnable vs fixed decay?")
fixed_f1 = all_ablation_results['Context_FixedDecay']['mean_f1']
print(f"  Learnable decay:  {full_f1:.3f}")
print(f"  Fixed decay:      {fixed_f1:.3f}")
print(f"  Delta:            {full_f1 - fixed_f1:+.3f}")

print(f"\nRQ4 — Turn-taking signals?")
nott_f1 = all_ablation_results['Context_NoTurnTaking']['mean_f1']
print(f"  With turn-taking: {full_f1:.3f}")
print(f"  Without:          {nott_f1:.3f}")
print(f"  Delta:            {full_f1 - nott_f1:+.3f}")

print(f"\nAll results saved to Drive: {DRIVE_ROOT}")
backup_cache()
print("\n✅ Article #2 pipeline complete.")